<a href="https://colab.research.google.com/github/hemanth2406/moshi/blob/main/moshi_telugu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
# Install the libraries if needed
!pip install datasets soundfile huggingface_hub

from datasets import load_dataset
from huggingface_hub import notebook_login

# Recommended: Log in to Hugging Face to avoid rate limits and warnings
# Running this will prompt you to enter your Hugging Face Access Token
notebook_login()

### 1. Load the Telugu Train Split
We explicitly select the `'train'` split of the `'telugu'` configuration of the `ai4bharat/IndicVoices` dataset.

In [ ]:
from huggingface_hub import login

login()

from datasets import load_dataset

print("Initializing streaming dataset...")
print("Starting dataset load...")

from datasets import load_dataset
from collections import Counter
ds = load_dataset("ai4bharat/IndicVoices", "telugu", split="train", streaming=True)
counts = Counter()
conv_rows = []
for i, row in enumerate(ds):
    sc = str(row.get("scenario") or "")
    counts[sc] += 1
    if sc.lower() == "conversation" and len(conv_rows) < 100:
        conv_rows.append(row)
    if len(conv_rows) >= 100 and i > 3000:
        break
print("scenario counts (sample):", counts)
print("conversation clips kept:", len(conv_rows))
print("example:", conv_rows[0]["scenario"], conv_rows[0]["duration"])

2. Build stereo on Drive (left = agent, right = user)
Pair clip 0 with 1, 2 with 3, … so each file has two different speakers.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, os, numpy as np, soundfile as sf, torch, torchaudio
out = "/content/drive/MyDrive/moshi_telugu/stereo_mvp"
os.makedirs(out, exist_ok=True)
def to_24k_mono(row):
    audio = row.get("audio_filepath") or row["audio"]
    wav = torch.as_tensor(audio["array"], dtype=torch.float32)
    if wav.ndim == 2:
        wav = wav.mean(0)
    sr = audio["sampling_rate"]
    if sr != 24000:
        wav = torchaudio.functional.resample(wav.unsqueeze(0), sr, 24000).squeeze(0)
    return wav.numpy()
jsonl_train, jsonl_eval = [], []
for n, i in enumerate(range(0, 80, 2)):          # 40 train files
    a, b = to_24k_mono(conv_rows[i]), to_24k_mono(conv_rows[i + 1])
    t = max(len(a), len(b))
    a, b = np.pad(a, (0, t - len(a))), np.pad(b, (0, t - len(b)))
    stereo = np.stack([a, b], axis=1)            # left=agent, right=user
    path = f"{out}/train_{n:03d}.wav"
    sf.write(path, stereo, 24000)
    dur = t / 24000
    jsonl_train.append({"path": path, "duration": dur})
    with open(path.replace(".wav", ".json"), "w", encoding="utf-8") as f:
        json.dump({"alignments": [["<utt>", [0.0, dur], "SPEAKER_MAIN"]]}, f)
for n, i in enumerate(range(80, 100, 2)):        # ~10 eval files (hold-out)
    a, b = to_24k_mono(conv_rows[i]), to_24k_mono(conv_rows[i + 1])
    t = max(len(a), len(b))
    a, b = np.pad(a, (0, t - len(a))), np.pad(b, (0, t - len(b)))
    stereo = np.stack([a, b], axis=1)
    path = f"{out}/eval_{n:03d}.wav"
    sf.write(path, stereo, 24000)
    dur = t / 24000
    jsonl_eval.append({"path": path, "duration": dur})
    with open(path.replace(".wav", ".json"), "w", encoding="utf-8") as f:
        json.dump({"alignments": [["<utt>", [0.0, dur], "SPEAKER_MAIN"]]}, f)
def write_jsonl(name, rows):
    with open(f"{out}/{name}", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
write_jsonl("train.jsonl", jsonl_train)
write_jsonl("eval.jsonl", jsonl_eval)
print("wrote", out, "train", len(jsonl_train), "eval", len(jsonl_eval))

### 3. Extract Mimi Codec Tokens
Below is the code to load the pretrained Mimi model, resample the audio to the required sample rate (Mimi typically uses 24kHz), and extract the audio tokens.

In [ ]:
import sys
import os

# Fallback installation to resolve the CUDA 13 vs CUDA 12 mismatch on Google Colab
try:
    import torch
    import torchaudio
except OSError as e:
    print("CUDA/binary mismatch detected. Re-installing CUDA-12 compatible torch/torchaudio and moshi...")
    !pip install --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu124
    !pip install --force-reinstall moshi
    import site
    from importlib import reload
    reload(site)
    import torch
    import torchaudio

# Ensure moshi is installed and loaded
try:
    from moshi.models import getters
except ImportError:
    print("Moshi module not found. Installing...")
    !pip install -q moshi
    import site
    from importlib import reload
    reload(site)
    from moshi.models import getters

# Load the pretrained Mimi codec model
device = "cuda" if torch.cuda.is_available() else "cpu"
mimi_model = getters.get_mimi()
mimi_model.to(device)
mimi_model.eval()

def encode_audio_to_mimi(audio_array, sampling_rate):
    # Convert numpy array to torch tensor
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0) # [1, samples]

    # Mimi expects 24kHz audio. Resample if necessary
    target_sample_rate = 24000
    if sampling_rate != target_sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    waveform = waveform.unsqueeze(0).to(device) # Shape: [1, 1, samples] (batch, channels, samples)

    with torch.no_grad():
        # Encode the waveform to discrete codes
        codes = mimi_model.encode(waveform)
        # codes shape: [batch, codebooks, timesteps]
    return codes.squeeze(0).cpu() # [codebooks, timesteps]

# Use the filtered sample
audio_data = sample['audio_filepath']['array'] if 'audio_filepath' in sample and isinstance(sample['audio_filepath'], dict) else sample['audio']['array']
sr = sample['audio_filepath']['sampling_rate'] if 'audio_filepath' in sample and isinstance(sample['audio_filepath'], dict) else sample['audio']['sampling_rate']

tokens = encode_audio_to_mimi(audio_data, sr)
print(f"Extracted Mimi tokens shape: {tokens.shape} (Codebooks: {tokens.shape[0]}, Timesteps: {tokens.shape[1]})")